# Goal 2 — Implementação e carregamento em MongoDB

Este notebook cria seis coleções a partir dos dados limpos do MovieLens: `movies`, `users`, `ratings`, `tags`, `genres` e `movie_genres`. As primeiras quatro seguem os exemplos de `Goal1/MongoDB`. As duas últimas preservam o catálogo de géneros e as ligações filme–género presentes nos CSV.

Cada filme mantém a lista de géneros do exemplo original. O notebook confirma que essa lista coincide com as ligações em `movie_genres` e os nomes em `genres`. A lista embebida permite obter os géneros ao ler o filme; as ligações permitem consultar a mesma relação por identificador.

**Como executar**

1. Abra o notebook na raiz de `BDAProject` e selecione um kernel Python. Se necessário, instale o driver com `%pip install pymongo` numa célula desse kernel.
2. Inicie o MongoDB. A base predefinida é bda_movielens. Confirme a ligação na configuração abaixo. Os ficheiros em `data/processed` são produzidos por `preparacao_dados.ipynb`.
3. Execute as células pela ordem indicada. À semelhança dos TP, cada execução elimina e recria as seis coleções deste notebook. Os documentos e índices existentes nessas coleções são eliminados.

In [9]:
%pip install pymongo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from collections import Counter, defaultdict
from csv import DictReader
from datetime import datetime
import json
import os   
from pathlib import Path
from pprint import pprint
from time import perf_counter

from pymongo import MongoClient

dados = Path("data/processed")
mongo_uri = os.getenv("BDA_MONGO_URI", "mongodb://localhost:27017/")
mongo_base = "bda_movielens"

tamanho_lote = 5000

## 1. Preparar os documentos

O notebook lê os ficheiros limpos sem repetir a amostragem. Os identificadores e anos são inteiros, os valores em falta ficam como `null` e as datas são interpretadas como UTC. Os documentos de filmes provêm do JSON e são conferidos com os CSV do modelo relacional.

In [11]:
def ler_csv(nome):
    with (dados / nome).open(encoding="utf-8-sig", newline="") as ficheiro:
        return list(DictReader(ficheiro))


def inteiro_opcional(valor):
    return int(valor) if valor not in ("", None) else None


with (dados / "movies_mongo.json").open(encoding="utf-8") as ficheiro:
    filmes = json.load(ficheiro)

for filme in filmes:
    # Também aceita ficheiros com identificadores serializados como 123.0.
    for campo in ("year", "imdbId", "tmdbId"):
        filme[campo] = inteiro_opcional(filme[campo])
    filme["_id"] = filme["movieId"]

utilizadores = [
    {"_id": int(linha["userId"]), "userId": int(linha["userId"])}
    for linha in ler_csv("users.csv")
]

avaliacoes = [
    {
        "_id": f"{linha['userId']}_{linha['movieId']}",
        "userId": int(linha["userId"]),
        "movieId": int(linha["movieId"]),
        "rating": float(linha["rating"]),
        "timestamp": datetime.fromisoformat(linha["timestamp"]),
    }
    for linha in ler_csv("ratings.csv")
]

etiquetas = [
    {
        "userId": int(linha["userId"]),
        "movieId": int(linha["movieId"]),
        "tag": linha["tag"],
        "timestamp": datetime.fromisoformat(linha["timestamp"]),
    }
    for linha in ler_csv("tags.csv")
]

generos_docs = [
    {"_id": int(linha["genreId"]), "genreId": int(linha["genreId"]), "genre": linha["genre"]}
    for linha in ler_csv("genres.csv")
]

filmes_generos = [
    {
        "_id": f"{linha['movieId']}_{linha['genreId']}",
        "movieId": int(linha["movieId"]),
        "genreId": int(linha["genreId"]),
    }
    for linha in ler_csv("movie_genres.csv")
]

documentos = {
    "movies": filmes,
    "users": utilizadores,
    "ratings": avaliacoes,
    "tags": etiquetas,
    "genres": generos_docs,
    "movie_genres": filmes_generos,
}
print({nome: len(linhas) for nome, linhas in documentos.items()})
pprint(avaliacoes[0])

{'movies': 9742, 'users': 610, 'ratings': 100000, 'tags': 3683, 'genres': 19, 'movie_genres': 22050}
{'_id': '432_77866',
 'movieId': 77866,
 'rating': 4.5,
 'timestamp': datetime.datetime(2012, 4, 23, 0, 7, 21),
 'userId': 432}


In [12]:
generos = {doc["genreId"]: doc["genre"] for doc in generos_docs}
assert len(generos) == len(generos_docs)
assert len({doc["_id"] for doc in filmes_generos}) == len(filmes_generos)

generos_por_filme = defaultdict(list)
for ligacao in filmes_generos:
    generos_por_filme[ligacao["movieId"]].append(generos[ligacao["genreId"]])

filmes_csv = {}
for linha in ler_csv("movies.csv"):
    movie_id = int(linha["movieId"])
    filmes_csv[movie_id] = {
        "title": linha["title"],
        "year": inteiro_opcional(linha["year"]),
        "imdbId": inteiro_opcional(linha["imdbId"]),
        "tmdbId": inteiro_opcional(linha["tmdbId"]),
        "genres": sorted(generos_por_filme[movie_id]),
    }

filmes_json = {
    filme["movieId"]: {
        **{campo: filme[campo] for campo in ("title", "year", "imdbId", "tmdbId")},
        "genres": sorted(filme["genres"]),
    }
    for filme in filmes
}
assert len(filmes_json) == len(filmes) and filmes_json == filmes_csv
assert generos_por_filme.keys() <= filmes_csv.keys()
ids_utilizadores = {doc["userId"] for doc in utilizadores}
assert len(ids_utilizadores) == len(utilizadores)
assert len(avaliacoes) == 100_000
assert len({doc["_id"] for doc in avaliacoes}) == len(avaliacoes)

for nome in ("ratings", "tags"):
    assert all(doc["movieId"] in filmes_json for doc in documentos[nome])
    assert all(doc["userId"] in ids_utilizadores for doc in documentos[nome])

print("Fontes equivalentes. As referências de avaliações, etiquetas e géneros são válidas.")

Fontes equivalentes. As referências de avaliações, etiquetas e géneros são válidas.


## 2. Criar as coleções com validação

Em `movies`, `users` e `genres`, o `_id` é o identificador do registo. Em `ratings` usa-se `userId_movieId`, e em `movie_genres` usa-se `movieId_genreId`. A validação confirma que os campos do documento coincidem com a chave. O índice automático de `_id` impede duplicados. Em `tags`, o MongoDB gera o `_id`. Os índices para consultas ficam para o Goal 4.

In [13]:
inteiro = {"bsonType": "int", "minimum": 1}
inteiro_opcional_bson = {"bsonType": ["int", "null"]}
texto = {"bsonType": "string"}
data_bson = {"bsonType": "date"}

campos = {
    "movies": {
        "_id": inteiro,
        "movieId": inteiro,
        "title": {"bsonType": "string", "minLength": 1},
        "year": inteiro_opcional_bson,
        "genres": {"bsonType": "array", "items": texto},
        "imdbId": inteiro_opcional_bson,
        "tmdbId": inteiro_opcional_bson,
    },
    "users": {"_id": inteiro, "userId": inteiro},
    "ratings": {
        "_id": {"bsonType": "string", "pattern": "^[1-9][0-9]*_[1-9][0-9]*$"},
        "userId": inteiro,
        "movieId": inteiro,
        "rating": {"bsonType": "double", "minimum": 0.5, "maximum": 5.0},
        "timestamp": data_bson,
    },
    "genres": {
        "_id": inteiro,
        "genreId": inteiro,
        "genre": {"bsonType": "string", "minLength": 1},
    },
    "movie_genres": {
        "_id": {"bsonType": "string", "pattern": "^[1-9][0-9]*_[1-9][0-9]*$"},
        "movieId": inteiro,
        "genreId": inteiro,
    },
    "tags": {
        "userId": inteiro,
        "movieId": inteiro,
        "tag": texto,
        "timestamp": data_bson,
    },
}

chaves = {
    "movies": "$movieId",
    "users": "$userId",
    "genres": "$genreId",
    "movie_genres": {"$concat": [{"$toString": "$movieId"}, "_", {"$toString": "$genreId"}]},
    "ratings": {"$concat": [{"$toString": "$userId"}, "_", {"$toString": "$movieId"}]},
}

cliente = MongoClient(mongo_uri, serverSelectionTimeoutMS=5000)
cliente.admin.command("ping")
db = cliente[mongo_base]

for nome, propriedades in campos.items():
    db[nome].drop()
    validador = {
        "$jsonSchema": {
            "bsonType": "object",
            "required": list(propriedades),
            "properties": propriedades,
        },
    }
    if nome in chaves:
        validador["$expr"] = {"$eq": ["$_id", chaves[nome]]}

    db.create_collection(
        nome,
        validator=validador,
        validationLevel="strict",
        validationAction="error",
    )

print("Coleções criadas em", mongo_base)

Coleções criadas em bda_movielens


## 3. Medir o carregamento

O notebook envia os documentos em lotes de 5 000 com `insert_many`. O cronómetro mede o tempo de escrita por coleção e o total. A leitura dos ficheiros, a criação das coleções e as verificações ficam fora da medição.

In [14]:
tempos_mongo = {}
for nome, linhas in documentos.items():
    inicio = perf_counter()
    for i in range(0, len(linhas), tamanho_lote):
        db[nome].insert_many(linhas[i:i + tamanho_lote], ordered=True)
    tempos_mongo[nome] = perf_counter() - inicio
    print(f"{nome:8} {len(linhas):>7} documentos em {tempos_mongo[nome]:.4f} s")

print(f"Total MongoDB: {sum(tempos_mongo.values()):.4f} s")

movies      9742 documentos em 0.0584 s
users        610 documentos em 0.0032 s
ratings   100000 documentos em 0.7816 s
tags        3683 documentos em 0.0211 s
genres        19 documentos em 0.0007 s
movie_genres   22050 documentos em 0.1133 s
Total MongoDB: 0.9783 s


## 4. Verificar os dados no MongoDB

Conferimos as contagens das seis coleções, todas as ligações filme–género, os géneros embebidos e uma amostra dos filmes, utilizadores, avaliações e etiquetas. Os dados limpos são a referência para as verificações.

In [15]:
for nome, linhas in documentos.items():
    total = db[nome].count_documents({})
    assert total == len(linhas), (nome, total, len(linhas))
    print(f"{nome:8} ficheiro={len(linhas):>7} MongoDB={total:>7}")

generos_mongo = {
    doc["genreId"]: doc["genre"] for doc in db.genres.find()
}
assert generos_mongo == generos

ligacoes_mongo = {
    (doc["movieId"], doc["genreId"]) for doc in db.movie_genres.find()
}
assert ligacoes_mongo == {
    (doc["movieId"], doc["genreId"]) for doc in filmes_generos
}

generos_embebidos = {
    doc["movieId"]: sorted(doc["genres"])
    for doc in db.movies.find({}, {"movieId": 1, "genres": 1})
}
assert generos_embebidos == {
    movie_id: info["genres"] for movie_id, info in filmes_csv.items()
}
print("Catálogo, ligações e géneros embebidos verificados.")

filme_id = 1
assert db.movies.find_one({"_id": filme_id}) == next(
    filme for filme in filmes if filme["movieId"] == filme_id
)
assert db.users.find_one({"_id": utilizadores[0]["userId"]}) == utilizadores[0]
assert db.ratings.find_one({"_id": avaliacoes[0]["_id"]}) == avaliacoes[0]

filtro_filme = {"movieId": filme_id}
pares_esperados = Counter(
    (doc["userId"], doc["rating"], doc["timestamp"])
    for doc in avaliacoes if doc["movieId"] == filme_id
)
pares_mongo = Counter(
    (doc["userId"], doc["rating"], doc["timestamp"])
    for doc in db.ratings.find(filtro_filme)
)
assert pares_mongo == pares_esperados

etiquetas_esperadas = Counter(
    (doc["userId"], doc["tag"], doc["timestamp"])
    for doc in etiquetas if doc["movieId"] == filme_id
)
assert Counter(
    (doc["userId"], doc["tag"], doc["timestamp"])
    for doc in db.tags.find(filtro_filme)
) == etiquetas_esperadas

print(f"Filme {filme_id}: {pares_mongo.total()} avaliações e {etiquetas_esperadas.total()} etiquetas verificadas.")
for nome in documentos:
    print(f"Índices iniciais de {nome}: {list(db[nome].index_information())}")

cliente.close()

movies   ficheiro=   9742 MongoDB=   9742
users    ficheiro=    610 MongoDB=    610
ratings  ficheiro= 100000 MongoDB= 100000
tags     ficheiro=   3683 MongoDB=   3683
genres   ficheiro=     19 MongoDB=     19
movie_genres ficheiro=  22050 MongoDB=  22050
Catálogo, ligações e géneros embebidos verificados.
Filme 1: 214 avaliações e 3 etiquetas verificadas.
Índices iniciais de movies: ['_id_']
Índices iniciais de users: ['_id_']
Índices iniciais de ratings: ['_id_']
Índices iniciais de tags: ['_id_']
Índices iniciais de genres: ['_id_']
Índices iniciais de movie_genres: ['_id_']
